# 02 - EDA e Storytelling (11 perguntas do enunciado)

Este notebook responde as 11 perguntas do Datathon Passos Mágicos usando **apenas** as saídas já consolidadas:

- `data/silver/base_unificada_validada_enriquecida.csv`
- `data/gold/base_modelagem_risco.csv`

Fonte das perguntas: `POSTECH - Datathon - Fase 5 (2).pdf` (páginas 3 e 4).



In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (10, 5)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

silver_path = PROJECT_ROOT / "data" / "silver" / "base_unificada_validada_enriquecida.csv"
gold_path = PROJECT_ROOT / "data" / "gold" / "base_modelagem_risco.csv"

silver = pd.read_csv(silver_path, low_memory=False)
gold = pd.read_csv(gold_path, low_memory=False)

# Foco oficial do case: 2022-2024
silver_eda = silver[silver["ano_referencia"].between(2022, 2024)].copy()
gold_eda = gold[gold["ano_referencia"].between(2022, 2024)].copy()

print("Silver EDA shape:", silver_eda.shape)
print("Gold EDA shape:", gold_eda.shape)
print("Taxa média de risco (%):", round(gold_eda["target_risco"].mean() * 100, 2))



In [ ]:
# Helpers de categorização para leitura executiva

def add_ian_band(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # Escala prática em 0-10 para leitura de defasagem (proxy):
    # <=4 severa, (4,6] moderada, >6 adequada
    out["faixa_ian"] = pd.cut(
        out["ian"],
        bins=[-np.inf, 4, 6, np.inf],
        labels=["Severa", "Moderada", "Adequada"],
    )
    return out


def safe_qcut(series: pd.Series, label: str, q: int = 4) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    valid = s.dropna()
    if valid.nunique() <= 1:
        out = pd.Series(index=s.index, dtype="object")
        out.loc[valid.index] = f"{label}_Q1"
        return out

    q_eff = min(q, int(valid.nunique()))
    ranked = valid.rank(method="first")
    labels = [f"{label}_Q{i}" for i in range(1, q_eff + 1)]
    buckets = pd.qcut(ranked, q=q_eff, labels=labels)

    out = pd.Series(index=s.index, dtype="object")
    out.loc[valid.index] = buckets.astype(str)
    return out


def add_quartile_band(df: pd.DataFrame, col: str, label: str) -> pd.Series:
    return safe_qcut(df[col], label=label, q=4)




## 1) Adequação do nível (IAN)
**Pergunta:** Qual é o perfil geral de defasagem (IAN) e como ele evolui ao longo dos anos?



In [ ]:
q1 = add_ian_band(silver_eda)

tab_q1 = (
    q1.groupby(["ano_referencia", "faixa_ian"]).size().reset_index(name="alunos")
    .pivot(index="ano_referencia", columns="faixa_ian", values="alunos")
    .fillna(0)
    .astype(int)
)

display(tab_q1)

tab_q1_pct = tab_q1.div(tab_q1.sum(axis=1), axis=0) * 100
ax = tab_q1_pct.plot(kind="bar", stacked=True)
ax.set_ylabel("% de alunos")
ax.set_title("Distribuição de faixas de IAN por ano")
plt.show()



## 2) Desempenho acadêmico (IDA)
**Pergunta:** O IDA médio está melhorando, estagnado ou caindo ao longo de fases e anos?



In [ ]:
q2 = silver_eda.groupby(["ano_referencia", "fase"], as_index=False)["ida"].mean()

display(q2.pivot(index="fase", columns="ano_referencia", values="ida").round(2))

plt.figure(figsize=(11, 5))
sns.lineplot(data=q2, x="ano_referencia", y="ida", hue="fase", marker="o")
plt.title("IDA médio por fase e ano")
plt.ylabel("IDA médio")
plt.xlabel("Ano")
plt.show()



## 3) Engajamento nas atividades (IEG)
**Pergunta:** O IEG tem relação direta com IDA e IPV?



In [ ]:
q3 = silver_eda[["ieg", "ida", "ipv"]].dropna()

corr_q3 = q3.corr(numeric_only=True)
display(corr_q3.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.regplot(data=q3, x="ieg", y="ida", scatter_kws={"alpha": 0.3}, ax=axes[0])
axes[0].set_title("IEG vs IDA")
sns.regplot(data=q3, x="ieg", y="ipv", scatter_kws={"alpha": 0.3}, ax=axes[1])
axes[1].set_title("IEG vs IPV")
plt.tight_layout()
plt.show()


## 4) Autoavaliação (IAA)
**Pergunta:** O IAA é coerente com IDA e IEG?



In [ ]:
q4 = silver_eda[["iaa", "ida", "ieg"]].dropna().copy()
q4["faixa_iaa"] = add_quartile_band(q4, "iaa", "IAA")

tab_q4 = q4.groupby("faixa_iaa", as_index=False).agg(
    iaa_media=("iaa", "mean"),
    ida_media=("ida", "mean"),
    ieg_media=("ieg", "mean"),
    n=("iaa", "size"),
)

display(tab_q4.round(3))

plt.figure(figsize=(10, 4))
plot_df = tab_q4.melt(id_vars=["faixa_iaa", "n"], value_vars=["ida_media", "ieg_media"], var_name="indicador", value_name="valor")
sns.barplot(data=plot_df, x="faixa_iaa", y="valor", hue="indicador")
plt.title("Coerência entre IAA e desempenho/engajamento")
plt.ylabel("Média")
plt.xlabel("Faixa de IAA")
plt.show()



## 5) Aspectos psicossociais (IPS)
**Pergunta:** Há padrões de IPS que antecedem quedas de desempenho?



In [ ]:
q5 = gold_eda[["ips", "delta_ida_next", "target_risco"]].dropna(subset=["ips"]).copy()
q5["faixa_ips"] = add_quartile_band(q5, "ips", "IPS")

agg_q5 = q5.groupby("faixa_ips", as_index=False).agg(
    risco_medio=("target_risco", "mean"),
    delta_ida_medio=("delta_ida_next", "mean"),
    n=("target_risco", "size"),
)
agg_q5["risco_medio"] = agg_q5["risco_medio"] * 100

display(agg_q5.round(3))

fig, ax1 = plt.subplots(figsize=(10, 4))
sns.barplot(data=agg_q5, x="faixa_ips", y="risco_medio", color="#2a9d8f", ax=ax1)
ax1.set_ylabel("Risco médio (%)")
ax1.set_xlabel("Faixa de IPS")
ax1.set_title("IPS vs risco e queda futura de IDA")

ax2 = ax1.twinx()
sns.lineplot(data=agg_q5, x="faixa_ips", y="delta_ida_medio", marker="o", color="#e76f51", ax=ax2)
ax2.set_ylabel("Delta IDA futuro médio")

plt.show()



## 6) Aspectos psicopedagógicos (IPP)
**Pergunta:** O IPP confirma ou contradiz a defasagem observada em IAN?



In [ ]:
q6 = add_ian_band(silver_eda[["ian", "ipp"]].dropna())

tab_q6 = q6.groupby("faixa_ian", as_index=False).agg(
    ian_medio=("ian", "mean"),
    ipp_medio=("ipp", "mean"),
    n=("ipp", "size"),
)

display(tab_q6.round(3))

plt.figure(figsize=(9, 4))
sns.boxplot(data=q6, x="faixa_ian", y="ipp")
plt.title("Distribuição de IPP por faixa de IAN")
plt.xlabel("Faixa IAN")
plt.ylabel("IPP")
plt.show()



## 7) Ponto de virada (IPV)
**Pergunta:** Quais comportamentos mais influenciam o IPV ao longo do tempo?


In [ ]:
features_q7 = ["ida", "ieg", "iaa", "ips", "ipp", "ian", "defasagem", "inde"]
q7 = silver_eda[features_q7 + ["ipv", "ano_referencia"]].copy()
q7 = q7.dropna(subset=["ipv"])

X_q7 = q7[features_q7]
y_q7 = q7["ipv"]

imputer = SimpleImputer(strategy="median")
X_q7_imp = imputer.fit_transform(X_q7)

rf_reg = RandomForestRegressor(n_estimators=300, random_state=42)
rf_reg.fit(X_q7_imp, y_q7)

imp_q7 = pd.DataFrame({
    "feature": features_q7,
    "importancia": rf_reg.feature_importances_,
}).sort_values("importancia", ascending=False)

display(imp_q7)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=imp_q7.head(8), x="importancia", y="feature", ax=axes[0])
axes[0].set_title("Importância para explicar IPV")

pv_ano = q7.groupby("ano_referencia", as_index=False)["ipv"].mean()
sns.lineplot(data=pv_ano, x="ano_referencia", y="ipv", marker="o", ax=axes[1])
axes[1].set_title("Evolução do IPV médio")
axes[1].set_ylabel("IPV médio")
plt.tight_layout()
plt.show()



## 8) Multidimensionalidade dos indicadores
**Pergunta:** Quais combinações de IDA + IEG + IPS + IPP elevam mais o INDE?



In [ ]:
q8 = silver_eda[["ida", "ieg", "ips", "ipp", "inde"]].dropna().copy()
for c in ["ida", "ieg", "ips", "ipp"]:
    q8[f"{c}_nivel"] = np.where(q8[c] >= q8[c].median(), "Alto", "Baixo")

q8["combinacao"] = (
    "IDA_" + q8["ida_nivel"] + " | "
    + "IEG_" + q8["ieg_nivel"] + " | "
    + "IPS_" + q8["ips_nivel"] + " | "
    + "IPP_" + q8["ipp_nivel"]
)

agg_q8 = q8.groupby("combinacao", as_index=False).agg(
    inde_medio=("inde", "mean"),
    n=("inde", "size"),
).query("n >= 30").sort_values("inde_medio", ascending=False)

display(agg_q8.head(10).round(3))

plt.figure(figsize=(12, 5))
sns.barplot(data=agg_q8.head(10), y="combinacao", x="inde_medio", palette="Blues_r")
plt.title("Top combinações com maior INDE médio (n >= 30)")
plt.xlabel("INDE médio")
plt.ylabel("Combinação")
plt.show()



## 9) Previsão de risco com Machine Learning (visão exploratória)
**Pergunta:** Quais padrões de indicadores ajudam a identificar risco antes da queda?



In [ ]:
cols_q9 = ["target_risco", "ian", "ida", "ieg", "iaa", "ips", "ipp", "ipv", "inde", "defasagem", "delta_ida_next", "delta_ian_next"]
q9 = gold_eda[cols_q9].copy()

stats_q9 = q9.groupby("target_risco").mean(numeric_only=True).T
stats_q9.columns = ["sem_risco_0", "com_risco_1"]
stats_q9["dif_1_menos_0"] = stats_q9["com_risco_1"] - stats_q9["sem_risco_0"]
stats_q9 = stats_q9.sort_values("dif_1_menos_0", ascending=False)

display(stats_q9.round(3))

plt.figure(figsize=(10, 6))
plot_q9 = stats_q9[["dif_1_menos_0"]].sort_values("dif_1_menos_0")
sns.barplot(data=plot_q9.reset_index(), x="dif_1_menos_0", y="index", color="#264653")
plt.title("Sinais exploratórios associados ao risco (classe 1 - classe 0)")
plt.xlabel("Diferença de média")
plt.ylabel("Indicador")
plt.show()



## 10) Efetividade do programa por fase/pedra
**Pergunta:** Há melhora consistente nas fases (Quartzo, Ágata, Ametista, Topázio)?



In [ ]:
q10 = gold_eda[["ano_referencia", "pedra", "inde", "target_risco"]].dropna(subset=["pedra"]).copy()

agg_q10 = q10.groupby(["ano_referencia", "pedra"], as_index=False).agg(
    inde_medio=("inde", "mean"),
    risco_medio=("target_risco", "mean"),
    n=("target_risco", "size"),
)
agg_q10["risco_medio"] = agg_q10["risco_medio"] * 100

display(agg_q10.sort_values(["ano_referencia", "pedra"]).round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.lineplot(data=agg_q10, x="ano_referencia", y="inde_medio", hue="pedra", marker="o", ax=axes[0])
axes[0].set_title("INDE médio por pedra ao longo do tempo")

sns.lineplot(data=agg_q10, x="ano_referencia", y="risco_medio", hue="pedra", marker="o", ax=axes[1])
axes[1].set_title("Risco médio (%) por pedra ao longo do tempo")
axes[1].set_ylabel("Risco médio (%)")
plt.tight_layout()
plt.show()



## 11) Insights e criatividade
**Pergunta:** Que insights adicionais e sugestões práticas podem ser propostos?




In [ ]:
q11 = gold_eda[["ano_referencia", "ian", "ida", "ieg", "ips", "ipp", "inde", "target_risco"]].copy()

for c in ["ian", "ida", "ieg", "ips", "ipp", "inde"]:
    valid = q11[c].dropna()
    if valid.nunique() <= 1:
        q11[f"{c}_q"] = "Q1"
    else:
        q_eff = min(4, int(valid.nunique()))
        labels = [f"Q{i}" for i in range(1, q_eff + 1)]
        ranked = q11[c].rank(method="first")
        q11[f"{c}_q"] = pd.qcut(ranked, q=q_eff, labels=labels)

q11["segmento"] = (
    "IAN_" + q11["ian_q"].astype(str)
    + " | IDA_" + q11["ida_q"].astype(str)
    + " | IEG_" + q11["ieg_q"].astype(str)
)

seg = q11.groupby("segmento", as_index=False).agg(
    risco_medio=("target_risco", "mean"),
    inde_medio=("inde", "mean"),
    n=("target_risco", "size"),
)
seg = seg.query("n >= 40").sort_values("risco_medio", ascending=False)
seg["risco_medio"] = seg["risco_medio"] * 100

display(seg.head(10).round(3))

plt.figure(figsize=(12, 5))
sns.barplot(data=seg.head(10), y="segmento", x="risco_medio", palette="Reds_r")
plt.title("Top segmentos com maior risco médio (n >= 40)")
plt.xlabel("Risco médio (%)")
plt.ylabel("Segmento")
plt.show()

sugestoes = pd.DataFrame(
    {
        "frente": [
            "Monitoramento preventivo",
            "Ação acadêmica focada",
            "Apoio psicossocial/psicopedagógico",
            "Rotina de engajamento",
        ],
        "acao_pratica": [
            "Listar semanalmente top 15% por score de risco e abrir plano de ação individual.",
            "Trilhas de reforço para grupos com queda de IDA e baixa adequação (IAN).",
            "Priorizar acompanhamento IPS/IPP em segmentos com risco crescente.",
            "Acompanhar IEG quinzenal com gatilho para intervenção quando abaixo do percentil 25.",
        ],
    }
)

display(sugestoes)


